In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ============================================================
# CELL 1 — GPU CHECK
#
# WHAT HAPPENS:
# Checks that Kaggle provides an NVIDIA GPU.
# We will use GPU 0 for ProEmoTrans.
# ============================================================

!nvidia-smi

import torch

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Enable GPU in Kaggle Settings → Accelerator."
    )

print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0))
print("GPU count      :", torch.cuda.device_count())

print("\n✓ GPU is ready.")

Mon Sep 28 05:14:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================
# CELL 2 — DOWNLOAD OFFICIAL REPOSITORY
#
# WHAT HAPPENS:
# Downloads the complete ProEmoTrans GitHub repository.
# ============================================================

import os
import subprocess

PROJECT_ROOT = "/kaggle/working/ProEmoTrans"

if os.path.exists(PROJECT_ROOT):
    print("Repository already exists.")
else:
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/KunPunCN/ProEmoTrans.git",
            PROJECT_ROOT
        ],
        check=True
    )

print("✓ Official repository is ready.")

Cloning into '/kaggle/working/ProEmoTrans'...


✓ Official repository is ready.


In [3]:
# ============================================================
# CELL 3 — PROJECT DIRECTORY
#
# WHAT HAPPENS:
# Moves into the directory containing main_crf.py.
# ============================================================

%cd /kaggle/working/ProEmoTrans/ProEmoTrans

import os

print("Current directory:")
print(os.getcwd())

if not os.path.exists("main_crf.py"):
    raise FileNotFoundError("main_crf.py was not found.")

print("\n✓ Correct project directory.")

/kaggle/working/ProEmoTrans/ProEmoTrans
Current directory:
/kaggle/working/ProEmoTrans/ProEmoTrans

✓ Correct project directory.


In [4]:
# ============================================================
# CELL 4 — VERIFY PROJECT FILES
#
# WHAT HAPPENS:
# Makes sure the important source files supplied by the
# authors are actually present.
# ============================================================

required_files = [
    "avd.py",
    "csv_reader.py",
    "dataset.py",
    "input_instance.py",
    "main.py",
    "main_crf.py",
    "model.py",
    "model_crf.py",
    "run.sh",
    "utils.py",
]

for filename in required_files:
    if not os.path.isfile(filename):
        raise FileNotFoundError(
            f"Required project file missing: {filename}"
        )

print("=" * 70)
print("PROJECT FILE CHECK")
print("=" * 70)

for filename in required_files:
    print("✓", filename)

print("\n✓ All important source files found.")

PROJECT FILE CHECK
✓ avd.py
✓ csv_reader.py
✓ dataset.py
✓ input_instance.py
✓ main.py
✓ main_crf.py
✓ model.py
✓ model_crf.py
✓ run.sh
✓ utils.py

✓ All important source files found.


In [5]:
# ============================================================
# CELL 5 — DATASET CHECK
#
# WHAT HAPPENS:
# Verifies the processed datasets already included in the
# GitHub repository.
# ============================================================

DATA_DIR = "ERC_dataset"

required_data = [
    "dialogues_train.csv",
    "dialogues_val.csv",
    "dialogues_test.csv",

    "emory_train.csv",
    "emory_val.csv",
    "emory_test.csv",

    "iemocap_train.csv",
    "iemocap_val.csv",
    "iemocap_test.csv",

    "meld_train.csv",
    "meld_val.csv",
    "meld_test.csv",

    "dialogues_relation.json",
    "emory_relation.json",
    "iemocap_relation.json",
    "meld_relation.json",
]

missing = [
    f for f in required_data
    if not os.path.isfile(os.path.join(DATA_DIR, f))
]

if missing:
    print("Missing files:")
    for f in missing:
        print("✗", f)

    raise FileNotFoundError(
        "One or more required dataset files are missing."
    )

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

for f in required_data:
    print("✓", f)

print("\n✓ All required datasets found.")

DATASET CHECK
✓ dialogues_train.csv
✓ dialogues_val.csv
✓ dialogues_test.csv
✓ emory_train.csv
✓ emory_val.csv
✓ emory_test.csv
✓ iemocap_train.csv
✓ iemocap_val.csv
✓ iemocap_test.csv
✓ meld_train.csv
✓ meld_val.csv
✓ meld_test.csv
✓ dialogues_relation.json
✓ emory_relation.json
✓ iemocap_relation.json
✓ meld_relation.json

✓ All required datasets found.


In [6]:
# ============================================================
# CELL 6 — DISABLE UNNECESSARY OPTIONAL FRAMEWORKS
#
# WHAT HAPPENS:
# ProEmoTrans uses PyTorch.
#
# We do NOT need JAX/Flax/TensorFlow for this project.
# Disabling them prevents Transformers from trying to load
# incompatible optional packages.
# ============================================================

import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

print("USE_TF   =", os.environ["USE_TF"])
print("USE_FLAX =", os.environ["USE_FLAX"])

print("\n✓ Optional TensorFlow/JAX frameworks disabled.")

USE_TF   = 0
USE_FLAX = 0

✓ Optional TensorFlow/JAX frameworks disabled.


In [7]:
# ============================================================
# CELL 7 — SAFE DEPENDENCY INSTALLATION
#
# WHAT HAPPENS:
# Installs the packages required by ProEmoTrans.
#
# IMPORTANT:
# We deliberately DO NOT downgrade:
#   numpy
#   scipy
#   pandas
#   scikit-learn
#   torch
#
# Kaggle already provides them.
# ============================================================

!pip install -q --no-cache-dir \
    "transformers==4.36.2" \
    "tensorboardX==2.6.2.2" \
    "tqdm"

print("✓ Required packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 107.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 255.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 342.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 217.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
✓ Required packages installed.


In [8]:
# ============================================================
# CELL 8 — TRANSFORMERS TEST
#
# WHAT HAPPENS:
# Imports Transformers AFTER disabling optional JAX/Flax/TF.
#
# If this cell passes, the previous numpy/JAX error is fixed.
# ============================================================

import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import transformers

print("=" * 70)
print("TRANSFORMERS CHECK")
print("=" * 70)

print("Transformers version:", transformers.__version__)

print("\n✓ Transformers imported successfully.")

TRANSFORMERS CHECK
Transformers version: 4.36.2

✓ Transformers imported successfully.


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [9]:
# ============================================================
# CELL 9 — COMPLETE DEPENDENCY CHECK
#
# WHAT HAPPENS:
# Confirms that the libraries required by the project import
# correctly before we start expensive model loading.
# ============================================================

import torch
import transformers
import numpy
import pandas
import scipy
import sklearn
import tensorboardX
import tqdm

print("=" * 70)
print("DEPENDENCY CHECK")
print("=" * 70)

print("Python       :", __import__("sys").version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("NumPy        :", numpy.__version__)
print("Pandas       :", pandas.__version__)
print("SciPy        :", scipy.__version__)
print("Sklearn      :", sklearn.__version__)
print("TensorBoardX :", tensorboardX.__version__)

print("\nCUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n✓ Dependency check passed.")

DEPENDENCY CHECK
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 4.36.2
NumPy        : 2.0.2
Pandas       : 2.3.3
SciPy        : 1.16.3
Sklearn      : 1.6.1
TensorBoardX : 2.6.2.2

CUDA: True
GPU: Tesla T4

✓ Dependency check passed.


In [10]:
# ============================================================
# CELL 10 — PROEMOTRANS IMPORT TEST
#
# WHAT HAPPENS:
# Imports the actual research code.
#
# We test model_crf.py because this is the CRF/AVD version
# used for the main ProEmoTrans experiment.
# ============================================================

import sys
import os

PROJECT_DIR = "/kaggle/working/ProEmoTrans/ProEmoTrans"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Testing project modules...")

import avd
print("✓ avd.py")

import csv_reader
print("✓ csv_reader.py")

import input_instance
print("✓ input_instance.py")

import dataset
print("✓ dataset.py")

import model_crf
print("✓ model_crf.py")

import utils
print("✓ utils.py")

print("\n✓ ProEmoTrans imports successfully.")

Testing project modules...
✓ avd.py
✓ csv_reader.py
✓ input_instance.py
✓ dataset.py


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✓ model_crf.py
✓ utils.py

✓ ProEmoTrans imports successfully.


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [11]:
# ============================================================
# CELL 11 — BERT DOWNLOAD TEST
#
# WHAT HAPPENS:
# Downloads/caches bert-base-uncased.
#
# This is the encoder used by the repository.
# ============================================================

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-uncased"

print("=" * 70)
print("LOADING BERT")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✓ Tokenizer loaded.")

bert = AutoModel.from_pretrained(MODEL_NAME)

print("✓ BERT loaded.")
print("Hidden size:", bert.config.hidden_size)

LOADING BERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✓ BERT loaded.
Hidden size: 768


In [12]:
# ============================================================
# CELL 12 — BERT GPU TEST
#
# WHAT HAPPENS:
# Sends a sample sentence through BERT on GPU.
#
# This confirms that:
#     PyTorch → CUDA → BERT
# works before the real experiment starts.
# ============================================================

import torch

device = torch.device("cuda:0")

bert = bert.to(device)

text = "I am very happy today."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = bert(**inputs)

print("=" * 70)
print("BERT GPU TEST")
print("=" * 70)

print("Device :", device)
print("GPU    :", torch.cuda.get_device_name(0))
print("Output :", outputs.last_hidden_state.shape)

print("\n✓ BERT GPU test passed.")

BERT GPU TEST
Device : cuda:0
GPU    : Tesla T4
Output : torch.Size([1, 8, 768])

✓ BERT GPU test passed.


In [21]:
# ============================================================
# CELL 13 — CREATE OUTPUT DIRECTORIES
#
# WHAT HAPPENS:
# Creates all folders needed BEFORE training starts.
#
# This fixes the exact error:
#
# FileNotFoundError:
# logs/proemotrans_run.log
#
# We create:
#   logs/
#   checkpoints/
# ============================================================

import os

print("=" * 80)
print("📁 PREPARING OUTPUT DIRECTORIES")
print("=" * 80)

# Create logs directory
os.makedirs("logs", exist_ok=True)

# Create checkpoints directory
os.makedirs("checkpoints", exist_ok=True)

print("✓ logs/ directory ready")
print("✓ checkpoints/ directory ready")

print("\nCurrent project directory:")
print(os.getcwd())

print("\n✓ Output environment is ready.")

📁 PREPARING OUTPUT DIRECTORIES
✓ logs/ directory ready
✓ checkpoints/ directory ready

Current project directory:
/kaggle/working/ProEmoTrans/ProEmoTrans

✓ Output environment is ready.


In [22]:
# ============================================================
# CELL 14 — VERIFY LOGGING SYSTEM
#
# WHAT HAPPENS:
# Tests that Kaggle can actually create and write the log file.
#
# This prevents the FileNotFoundError before training.
# ============================================================

import os

log_path = os.path.abspath(
    "logs/proemotrans_run.log"
)

test_log = os.path.abspath(
    "logs/test_write.log"
)

print("=" * 80)
print("📝 LOGGING SYSTEM TEST")
print("=" * 80)

print("Log directory:")
print(os.path.dirname(log_path))

print("\nMain log:")
print(log_path)

# Test writing
with open(
    test_log,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "ProEmoTrans logging system test passed.\n"
    )

if not os.path.exists(test_log):

    raise RuntimeError(
        "Kaggle could not create the log file."
    )

print("\n✓ Log directory exists")
print("✓ Log file can be created")
print("✓ Log file can be written")

# Remove temporary test file
os.remove(test_log)

print("✓ Logging system is ready.")

📝 LOGGING SYSTEM TEST
Log directory:
/kaggle/working/ProEmoTrans/ProEmoTrans/logs

Main log:
/kaggle/working/ProEmoTrans/ProEmoTrans/logs/proemotrans_run.log

✓ Log directory exists
✓ Log file can be created
✓ Log file can be written
✓ Logging system is ready.


In [24]:
# ============================================================
# CELL 15 — PYTORCH CHECKPOINT COMPATIBILITY PATCH
#
# WHAT HAPPENS:
# The original project uses:
#
#     torch.load(checkpoint)
#
# Modern PyTorch versions use safer loading defaults.
#
# Because this repository saves the COMPLETE model object
# rather than only model weights, we explicitly use:
#
#     weights_only=False
#
# This is required for loading the repository's own checkpoint.
# ============================================================

import os

main_file = "main_crf.py"

if not os.path.exists(main_file):

    raise FileNotFoundError(
        "main_crf.py was not found."
    )

# Read original source
with open(
    main_file,
    "r",
    encoding="utf-8"
) as f:

    source = f.read()

# Make a backup
backup_file = "main_crf_original_backup.py"

if not os.path.exists(backup_file):

    with open(
        backup_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(source)

    print("✓ Original main_crf.py backed up.")

# Patch torch.load only if necessary
old_code = "model = torch.load(args.checkpoint_dir)"

new_code = (
    "model = torch.load("
    "args.checkpoint_dir, "
    "weights_only=False"
    ")"
)

if old_code in source:

    source = source.replace(
        old_code,
        new_code
    )

    with open(
        main_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(source)

    print("✓ PyTorch checkpoint compatibility patch applied.")

elif new_code in source:

    print(
        "✓ Checkpoint compatibility patch already exists."
    )

else:

    print(
        "⚠ torch.load line was not found exactly."
    )

print("\n✓ Checkpoint compatibility preparation complete.")

✓ Checkpoint compatibility patch already exists.

✓ Checkpoint compatibility preparation complete.


In [27]:
# ============================================================
# CELL 16 — VERIFY PYTORCH PATCH
#
# WHAT HAPPENS:
# Confirms that main_crf.py now contains the safe checkpoint
# loading option.
# ============================================================

with open(
    "main_crf.py",
    "r",
    encoding="utf-8"
) as f:

    source = f.read()

if "weights_only=False" in source:

    print("=" * 80)
    print("✅ CHECKPOINT PATCH VERIFIED")
    print("=" * 80)

    print(
        "main_crf.py now uses weights_only=False "
        "when loading the saved model."
    )

else:

    raise RuntimeError(
        "Checkpoint compatibility patch was not applied."
    )

✅ CHECKPOINT PATCH VERIFIED
main_crf.py now uses weights_only=False when loading the saved model.


In [28]:
# ============================================================
# CELL 17 — FINAL GPU CHECK
#
# WHAT HAPPENS:
# Performs one final GPU check immediately before training.
#
# We want:
#   CUDA = True
#   Tesla T4
#   enough free memory
# ============================================================

import torch

print("=" * 80)
print("🎮 FINAL GPU CHECK")
print("=" * 80)

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available. "
        "Enable GPU in Kaggle Settings."
    )

gpu_name = torch.cuda.get_device_name(0)

free_memory, total_memory = torch.cuda.mem_get_info(0)

free_gb = free_memory / (1024 ** 3)
total_gb = total_memory / (1024 ** 3)

print("CUDA available :", torch.cuda.is_available())
print("GPU            :", gpu_name)
print("GPU ID         : 0")
print(f"Free memory    : {free_gb:.2f} GB")
print(f"Total memory   : {total_gb:.2f} GB")

if free_gb < 4:

    raise RuntimeError(
        "GPU has less than 4 GB free memory. "
        "Restart the Kaggle session."
    )

print("\n✓ GPU ready")
print("✓ Memory sufficient")
print("✓ Ready for training")

🎮 FINAL GPU CHECK
CUDA available : True
GPU            : Tesla T4
GPU ID         : 0
Free memory    : 13.97 GB
Total memory   : 14.56 GB

✓ GPU ready
✓ Memory sufficient
✓ Ready for training


In [29]:
# ============================================================
# CELL 18 — FINAL PROJECT STRUCTURE CHECK
#
# WHAT HAPPENS:
# Makes sure everything needed for training is present.
# ============================================================

import os

print("=" * 80)
print("🔍 FINAL PROJECT CHECK")
print("=" * 80)

required = [
    "main_crf.py",
    "model_crf.py",
    "dataset.py",
    "avd.py",
    "csv_reader.py",
    "input_instance.py",
    "utils.py",
    "ERC_dataset/dialogues_train.csv",
    "ERC_dataset/emory_train.csv",
    "ERC_dataset/iemocap_train.csv",
    "ERC_dataset/meld_train.csv",
    "logs",
    "checkpoints",
]

missing = []

for item in required:

    if not os.path.exists(item):

        missing.append(item)

    else:

        print("✓", item)

if missing:

    print("\n❌ Missing:")
    
    for item in missing:
        print("   ", item)

    raise FileNotFoundError(
        "Required project files/directories are missing."
    )

print("\n✓ COMPLETE PROJECT CHECK PASSED")

🔍 FINAL PROJECT CHECK
✓ main_crf.py
✓ model_crf.py
✓ dataset.py
✓ avd.py
✓ csv_reader.py
✓ input_instance.py
✓ utils.py
✓ ERC_dataset/dialogues_train.csv
✓ ERC_dataset/emory_train.csv
✓ ERC_dataset/iemocap_train.csv
✓ ERC_dataset/meld_train.csv
✓ logs
✓ checkpoints

✓ COMPLETE PROJECT CHECK PASSED


In [30]:
# ============================================================
# CELL 19 — BUILD TRAINING COMMAND
#
# WHAT HAPPENS:
# Creates the exact command used to run ProEmoTrans.
#
# IMPORTANT:
# This cell does NOT start training.
# ============================================================

import sys

command = [
    sys.executable,
    "-u",
    "main_crf.py",

    # GPU
    "--gpu_available", "0",

    # Few-shot / unseen configuration
    "--unseen", "15",
    "--k", "2",
    "--seed", "7",

    # Batch sizes
    "--train_batch_size", "4",
    "--evaluate_batch_size", "4",

    # Training
    "--epochs", "10",
    "--lr", "1e-5",
    "--warm_up", "100",

    # BERT
    "--pretrained_model_name_or_path",
    "bert-base-uncased",

    # Auto matching
    "--add_auto_match",
    "False",
]

print("=" * 80)
print("⚙️ PROEMOTRANS TRAINING CONFIGURATION")
print("=" * 80)

print("\nGPU:")
print("   GPU ID: 0")
print("   GPU: Tesla T4")

print("\nModel:")
print("   bert-base-uncased")

print("\nExperiment:")
print("   unseen = 15")
print("   k = 2")
print("   seed = 7")

print("\nTraining:")
print("   batch size = 4")
print("   evaluation batch size = 4")
print("   epochs = 10")
print("   learning rate = 1e-5")
print("   warm-up = 100")

print("\n" + "=" * 80)
print("COMMAND")
print("=" * 80)

print(
    " ".join(command)
)

print("\n✓ Command prepared.")

⚙️ PROEMOTRANS TRAINING CONFIGURATION

GPU:
   GPU ID: 0
   GPU: Tesla T4

Model:
   bert-base-uncased

Experiment:
   unseen = 15
   k = 2
   seed = 7

Training:
   batch size = 4
   evaluation batch size = 4
   epochs = 10
   learning rate = 1e-5
   warm-up = 100

COMMAND
/usr/bin/python3 -u main_crf.py --gpu_available 0 --unseen 15 --k 2 --seed 7 --train_batch_size 4 --evaluate_batch_size 4 --epochs 10 --lr 1e-5 --warm_up 100 --pretrained_model_name_or_path bert-base-uncased --add_auto_match False

✓ Command prepared.


In [31]:
# ============================================================
# CELL 20 — RUN PROEMOTRANS
#
# WHAT HAPPENS:
#
# 1. Creates the log file
# 2. Starts main_crf.py
# 3. Shows LIVE output
# 4. Shows tqdm training progress
# 5. Saves complete output to:
#
#       logs/proemotrans_run.log
#
# 6. Shows runtime
# 7. Reports success/failure
#
# DO NOT INTERRUPT THIS CELL WHILE TRAINING.
# ============================================================

import os
import sys
import subprocess
import time

# ------------------------------------------------------------
# Make absolutely sure directories exist
# ------------------------------------------------------------

os.makedirs(
    "logs",
    exist_ok=True
)

os.makedirs(
    "checkpoints",
    exist_ok=True
)

# ------------------------------------------------------------
# Absolute log path
# ------------------------------------------------------------

log_path = os.path.abspath(
    "logs/proemotrans_run.log"
)

print("=" * 90)
print("🚀 STARTING PROEMOTRANS")
print("=" * 90)

print("\n📁 Working directory:")
print(os.getcwd())

print("\n📝 Log file:")
print(log_path)

print("\n🎮 GPU:")
print("Tesla T4 — CUDA device 0")

print("\n🧠 Model:")
print("bert-base-uncased")

print("\n⚙️ Experiment:")
print("Unseen classes : 15")
print("K              : 2")
print("Seed           : 7")
print("Epochs         : 10")
print("Batch size     : 4")
print("Learning rate  : 1e-5")

print("\n" + "=" * 90)
print("📋 COMMAND")
print("=" * 90)

print(
    " ".join(command)
)

print("\n" + "=" * 90)
print("⏳ LOADING / TRAINING OUTPUT")
print("=" * 90)

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

environment = os.environ.copy()

environment["PYTHONUNBUFFERED"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"
environment["USE_TF"] = "0"
environment["USE_FLAX"] = "0"

# Encourage visible tqdm output
environment["TQDM_DISABLE"] = "0"

# ------------------------------------------------------------
# Start timer
# ------------------------------------------------------------

start_time = time.time()

# ------------------------------------------------------------
# Start process
# ------------------------------------------------------------

process = subprocess.Popen(
    command,

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env=environment
)

# ------------------------------------------------------------
# Save + display output simultaneously
# ------------------------------------------------------------

with open(
    log_path,
    "w",
    encoding="utf-8"
) as log_file:

    for line in process.stdout:

        # Show live output in Kaggle
        print(
            line,
            end="",
            flush=True
        )

        # Save output
        log_file.write(line)
        log_file.flush()

# ------------------------------------------------------------
# Wait for process
# ------------------------------------------------------------

return_code = process.wait()

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n")
print("=" * 90)

if return_code == 0:

    print("🎉 PROEMOTRANS FINISHED SUCCESSFULLY!")

    print(
        f"⏱️ Total runtime: "
        f"{elapsed / 60:.2f} minutes"
    )

else:

    print("❌ PROEMOTRANS ENDED WITH AN ERROR")

    print(
        "Return code:",
        return_code
    )

    print(
        "\nThe complete error/output is saved here:"
    )

    print(log_path)

print("\n📄 Log saved:")
print(log_path)

print("=" * 90)

🚀 STARTING PROEMOTRANS

📁 Working directory:
/kaggle/working/ProEmoTrans/ProEmoTrans

📝 Log file:
/kaggle/working/ProEmoTrans/ProEmoTrans/logs/proemotrans_run.log

🎮 GPU:
Tesla T4 — CUDA device 0

🧠 Model:
bert-base-uncased

⚙️ Experiment:
Unseen classes : 15
K              : 2
Seed           : 7
Epochs         : 10
Batch size     : 4
Learning rate  : 1e-5

📋 COMMAND
/usr/bin/python3 -u main_crf.py --gpu_available 0 --unseen 15 --k 2 --seed 7 --train_batch_size 4 --evaluate_batch_size 4 --epochs 10 --lr 1e-5 --warm_up 100 --pretrained_model_name_or_path bert-base-uncased --add_auto_match False

⏳ LOADING / TRAINING OUTPUT
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._registe

In [33]:
# ============================================================
# CELL 21 — READ TRAINING RESULTS
#
# WHAT HAPPENS:
# Reads the saved training log and extracts:
#   [dev]
#   [test]
#   precision
#   recall
#   f1
#
# It NEVER crashes just because no result was found.
# ============================================================

import os

log_path = os.path.abspath(
    "logs/proemotrans_run.log"
)

print("=" * 90)
print("📊 PROEMOTRANS RESULTS")
print("=" * 90)

if not os.path.exists(log_path):

    print(
        "⚠ Training log does not exist yet."
    )

    print(
        "Run Cell 20 first."
    )

else:

    with open(
        log_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        lines = f.readlines()

    matches = []

    for line in lines:

        lower = line.lower()

        if (
            "[dev]" in lower
            or "[test]" in lower
            or "f1 score" in lower
            or "precision" in lower
            or "recall" in lower
        ):

            matches.append(
                line.strip()
            )

    if matches:

        print(
            "\n🎯 DETECTED EVALUATION RESULTS:\n"
        )

        for line in matches:

            print(line)

    else:

        print(
            "\n⚠ No evaluation result lines detected yet."
        )

        print(
            "\nThe complete log is available at:"
        )

        print(log_path)

print("\n✓ Result-reading cell completed.")

📊 PROEMOTRANS RESULTS

🎯 DETECTED EVALUATION RESULTS:

[dev][sim] (macro) final precision: 34.30, recall: 18.34, f1 score: 20.95
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
[test][sim] (macro) final precision: 9.27, recall: 25.50, f1 score: 12.66
[dev][sim] (macro) final precision: 34.59, recall: 32.94, f1 score: 32.79
[test][sim] (macro) final precision: 11.11, recall: 30.88, f1 score: 14.88
[dev][sim] (macro) final precision: 47.12, recall: 21.70, f1 score: 28.47
[test][sim] (macro) final precision: 17.32, recall: 16.71, f1 score: 15.73
[dev][sim] (macro) final precision: 47.17, recall: 28.01, f1 score: 32.68
[test][sim] (macro) final precision: 16.58, recall: 19.55, f1 score: 15.97
[dev][sim] (macro) final precision: 40.38, recall: 30.18, f1 score: 32.97
[test][sim] (macro) final precis

In [34]:
# ============================================================
# CELL 22 — CHECK TRAINED MODEL
#
# WHAT HAPPENS:
# Searches checkpoints/ for the trained ProEmoTrans model.
# ============================================================

import os

print("=" * 90)
print("💾 CHECKING MODEL CHECKPOINTS")
print("=" * 90)

checkpoint_files = []

checkpoint_dir = "checkpoints"

if os.path.exists(checkpoint_dir):

    for root, dirs, files in os.walk(
        checkpoint_dir
    ):

        for file in files:

            path = os.path.join(
                root,
                file
            )

            size_mb = (
                os.path.getsize(path)
                / (1024 ** 2)
            )

            checkpoint_files.append(
                path
            )

            print(
                f"✓ {path}"
            )

            print(
                f"  Size: {size_mb:.2f} MB"
            )

if not checkpoint_files:

    print(
        "\n⚠ No checkpoint found."
    )

    print(
        "This normally means training did not finish "
        "or the model was not saved."
    )

else:

    print(
        f"\n🎉 Found {len(checkpoint_files)} checkpoint(s)."
    )

print("\n✓ Checkpoint check completed.")

💾 CHECKING MODEL CHECKPOINTS
✓ checkpoints/emory_dialogues_unseen_15.pth
  Size: 423.11 MB

🎉 Found 1 checkpoint(s).

✓ Checkpoint check completed.


In [35]:
# ============================================================
# CELL 23 — SAVE FINAL EXPERIMENT SUMMARY
#
# WHAT HAPPENS:
# Creates a complete experiment summary.
#
# This cell:
#   ✓ creates logs/
#   ✓ never crashes if matches is undefined
#   ✓ records environment
#   ✓ records GPU
#   ✓ records model
#   ✓ records training settings
#   ✓ records evaluation results
#   ✓ records checkpoint information
# ============================================================

import os
import sys
import torch
import numpy as np
import transformers

# ------------------------------------------------------------
# Ensure output directory exists
# ------------------------------------------------------------

os.makedirs(
    "logs",
    exist_ok=True
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

log_path = os.path.abspath(
    "logs/proemotrans_run.log"
)

summary_path = os.path.abspath(
    "logs/experiment_summary.txt"
)

# ------------------------------------------------------------
# Read training log
# ------------------------------------------------------------

log_lines = []

if os.path.exists(log_path):

    with open(
        log_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        log_lines = f.readlines()

# ------------------------------------------------------------
# Extract results
# ------------------------------------------------------------

matches = []

for line in log_lines:

    lower = line.lower()

    if (
        "[dev]" in lower
        or "[test]" in lower
        or "f1 score" in lower
        or "precision" in lower
        or "recall" in lower
    ):

        matches.append(
            line.strip()
        )

# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    free_memory, total_memory = (
        torch.cuda.mem_get_info(0)
    )

    free_gb = (
        free_memory / (1024 ** 3)
    )

    total_gb = (
        total_memory / (1024 ** 3)
    )

else:

    gpu_name = "CPU"
    free_gb = 0
    total_gb = 0

# ------------------------------------------------------------
# Find checkpoints
# ------------------------------------------------------------

checkpoint_files = []

if os.path.exists("checkpoints"):

    for root, dirs, files in os.walk(
        "checkpoints"
    ):

        for file in files:

            checkpoint_files.append(
                os.path.join(
                    root,
                    file
                )
            )

# ------------------------------------------------------------
# Write summary
# ------------------------------------------------------------

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "ProEmoTrans Experiment Summary\n"
    )

    f.write(
        "=" * 80 + "\n\n"
    )

    # Environment
    f.write(
        "ENVIRONMENT\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        f"Python: {sys.version.split()[0]}\n"
    )

    f.write(
        f"PyTorch: {torch.__version__}\n"
    )

    f.write(
        f"Transformers: {transformers.__version__}\n"
    )

    f.write(
        f"NumPy: {np.__version__}\n"
    )

    f.write(
        f"CUDA available: "
        f"{torch.cuda.is_available()}\n"
    )

    f.write(
        f"GPU: {gpu_name}\n"
    )

    f.write(
        f"GPU total memory: "
        f"{total_gb:.2f} GB\n"
    )

    f.write(
        f"GPU free memory: "
        f"{free_gb:.2f} GB\n"
    )

    f.write("\n")

    # Model
    f.write(
        "MODEL\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        "Model: bert-base-uncased\n"
    )

    f.write(
        "GPU ID: 0\n"
    )

    f.write(
        "Unseen: 15\n"
    )

    f.write(
        "K: 2\n"
    )

    f.write(
        "Seed: 7\n"
    )

    f.write(
        "Train batch size: 4\n"
    )

    f.write(
        "Evaluation batch size: 4\n"
    )

    f.write(
        "Epochs: 10\n"
    )

    f.write(
        "Learning rate: 1e-5\n"
    )

    f.write(
        "Warm-up: 100\n"
    )

    f.write(
        "Auto match: False\n"
    )

    f.write("\n")

    # Results
    f.write(
        "EVALUATION RESULTS\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    if matches:

        for line in matches:

            f.write(
                line + "\n"
            )

    else:

        f.write(
            "No automatically detected evaluation output.\n"
        )

    f.write("\n")

    # Checkpoints
    f.write(
        "CHECKPOINTS\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    if checkpoint_files:

        for path in checkpoint_files:

            size_mb = (
                os.path.getsize(path)
                / (1024 ** 2)
            )

            f.write(
                f"{path} "
                f"({size_mb:.2f} MB)\n"
            )

    else:

        f.write(
            "No checkpoint files detected.\n"
        )

print("=" * 90)
print("✅ EXPERIMENT SUMMARY SAVED")
print("=" * 90)

print(
    "\nSummary:"
)

print(
    summary_path
)

print(
    "\nTraining log:"
)

print(
    log_path
)

print(
    f"\nEvaluation lines detected: "
    f"{len(matches)}"
)

print(
    f"Checkpoints found: "
    f"{len(checkpoint_files)}"
)

print("\n✓ Cell completed successfully.")

✅ EXPERIMENT SUMMARY SAVED

Summary:
/kaggle/working/ProEmoTrans/ProEmoTrans/logs/experiment_summary.txt

Training log:
/kaggle/working/ProEmoTrans/ProEmoTrans/logs/proemotrans_run.log

Evaluation lines detected: 23
Checkpoints found: 1

✓ Cell completed successfully.
